# 03. miRNA–mRNA Integration

This notebook performs the **main downstream miRNA–mRNA integration** using the revised differential-expression results.

## Inputs
- `results/DEM_exploratory_candidates.csv`
- `results/DEG_exploratory_candidates.csv`
- `results/miRNA_FDR_significant.csv`
- miRDB v6.0
- TargetScanHuman v8.0
- miRTarBase v9.0

## Interpretation
- BH-FDR significant miRNA(s) are marked separately.
- The broader miRNA–mRNA network is exploratory.
- Database support does not establish regulation in these cord-blood samples.


In [9]:
from pathlib import Path
import gzip
import pandas as pd
import numpy as np
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd.parents[1] if cwd.name == "revision" else cwd

RESULTS_DIR = PROJECT_ROOT / "results"
DATA_DIR = PROJECT_ROOT / "data"

# Current project structure
REF_DIR = DATA_DIR / "reference" / "mirna_targets"

MIRDB_FILE = REF_DIR / "miRDB_v6" / "miRDB_v6.0_prediction_result.txt.gz"
TARGETSCAN_DIR = REF_DIR / "TargetScan_v8"
MIRTARBASE_FILE = REF_DIR / "miRTarBase_v9" / "miRTarBase_MTI.xlsx"

DEM_FILE = RESULTS_DIR / "DEM_exploratory_candidates.csv"
DEG_FILE = RESULTS_DIR / "DEG_exploratory_candidates.csv"
FDR_MIRNA_FILE = RESULTS_DIR / "miRNA_FDR_significant.csv"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RESULTS_DIR  =", RESULTS_DIR)
print("REF_DIR      =", REF_DIR)


PROJECT_ROOT = /Users/jihopark/Desktop/MCDA_revision_final
RESULTS_DIR  = /Users/jihopark/Desktop/MCDA_revision_final/results
REF_DIR      = /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets


## 1. Check inputs

In [10]:
for p in [DEM_FILE, DEG_FILE, FDR_MIRNA_FILE, MIRDB_FILE]:
    print(("FOUND" if p.exists() else "MISSING"), "->", p)

print("\nTargetScan files:")
for p in TARGETSCAN_DIR.glob("*"):
    print(" -", p.name)

print("\nmiRTarBase:",
      "FOUND" if MIRTARBASE_FILE.exists() else "MISSING",
      "->", MIRTARBASE_FILE)

if not DEM_FILE.exists() or not DEG_FILE.exists() or not FDR_MIRNA_FILE.exists():
    raise FileNotFoundError("Run 02_differential_expression.ipynb first.")

if not MIRDB_FILE.exists():
    raise FileNotFoundError(MIRDB_FILE)


FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/results/DEM_exploratory_candidates.csv
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/results/DEG_exploratory_candidates.csv
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/results/miRNA_FDR_significant.csv
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets/miRDB_v6/miRDB_v6.0_prediction_result.txt.gz

TargetScan files:
 - Summary_Counts.default_predictions.txt.zip
 - miR_Family_Info.txt.zip

miRTarBase: FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets/miRTarBase_v9/miRTarBase_MTI.xlsx


## 2. Load revised candidate sets

In [11]:
dem = pd.read_csv(DEM_FILE)
deg = pd.read_csv(DEG_FILE)
fdr_mirna = pd.read_csv(FDR_MIRNA_FILE)

exploratory_mirnas = set(dem["miRNA"].dropna().astype(str).str.strip())
exploratory_genes = set(deg["Gene"].dropna().astype(str).str.strip())
confirmatory_mirnas = set(fdr_mirna["miRNA"].dropna().astype(str).str.strip())

print("Exploratory miRNAs:", len(exploratory_mirnas))
print("Exploratory mRNAs :", len(exploratory_genes))
print("FDR-significant miRNAs:", sorted(confirmatory_mirnas))


Exploratory miRNAs: 30
Exploratory mRNAs : 114
FDR-significant miRNAs: ['hsa-miR-1292-5p']


## 3. miRDB v6.0

In [12]:
def parse_mirdb(path):
    df = pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        header=None,
        comment="#",
        dtype=str
    )

    if df.shape[1] < 2:
        raise RuntimeError("Unexpected miRDB v6.0 format.")

    cols = ["miRNA", "Gene"]
    if df.shape[1] >= 3:
        cols.append("miRDB_score")
    cols += [f"extra_{i}" for i in range(len(cols), df.shape[1])]
    df.columns = cols

    df["miRNA"] = df["miRNA"].astype(str).str.strip()
    df["Gene"] = df["Gene"].astype(str).str.strip()

    out = df[
        df["miRNA"].isin(exploratory_mirnas) &
        df["Gene"].isin(exploratory_genes)
    ].copy()

    out["miRDB"] = True
    return out

mirdb_pairs = parse_mirdb(MIRDB_FILE)

print("miRDB candidate pairs:", len(mirdb_pairs))
display(mirdb_pairs.head())


miRDB candidate pairs: 0


,miRNA,Gene,miRDB_score,miRDB


## 4. TargetScanHuman v8.0

In [13]:
def read_table_auto(path):
    if path.suffix == ".zip":
        return pd.read_csv(
            path,
            sep="\t",
            compression="zip",
            low_memory=False
        )
    return pd.read_csv(
        path,
        sep="\t",
        low_memory=False
    )

def find_col(df, patterns):
    for c in df.columns:
        low = str(c).lower()
        if all(p.lower() in low for p in patterns):
            return c
    return None

summary_file = next(
    (p for p in TARGETSCAN_DIR.glob("*")
     if "summary_counts" in p.name.lower()),
    None
)

mirfam_file = next(
    (p for p in TARGETSCAN_DIR.glob("*")
     if "mir" in p.name.lower() and "family" in p.name.lower()),
    None
)

print("Summary Counts:", summary_file)
print("miR Family    :", mirfam_file)

if summary_file is None:
    raise FileNotFoundError("TargetScan Summary Counts file not found.")

ts = read_table_auto(summary_file)

gene_col = (
    find_col(ts, ["gene", "symbol"])
    or find_col(ts, ["gene symbol"])
)

mirfam_col = (
    find_col(ts, ["mirna", "family"])
    or find_col(ts, ["mir family"])
)

rep_mir_col = (
    find_col(ts, ["representative", "mirna"])
    or find_col(ts, ["representative mirna"])
)

species_col = find_col(ts, ["species", "id"])

if gene_col is None:
    raise RuntimeError("Could not identify TargetScan Gene Symbol column.")

if species_col is not None:
    ts = ts[ts[species_col].astype(str) == "9606"].copy()

ts["Gene"] = ts[gene_col].astype(str).str.strip()

if rep_mir_col is not None:
    ts["miRNA"] = ts[rep_mir_col].astype(str).str.strip()
else:
    ts["miRNA"] = ""

ts_pairs = ts[
    ts["miRNA"].isin(exploratory_mirnas) &
    ts["Gene"].isin(exploratory_genes)
].copy()

# Fallback family mapping
if len(ts_pairs) == 0 and mirfam_file is not None and mirfam_col is not None:
    mf = read_table_auto(mirfam_file)

    mf_family = (
        find_col(mf, ["mir", "family"])
        or find_col(mf, ["family"])
    )
    mf_mirbase = find_col(mf, ["mirbase", "id"])
    mf_species = find_col(mf, ["species", "id"])

    if mf_family and mf_mirbase:
        if mf_species:
            mf = mf[mf[mf_species].astype(str) == "9606"].copy()

        mf["miRNA"] = mf[mf_mirbase].astype(str).str.strip()
        mf["family_key"] = mf[mf_family].astype(str).str.strip()

        relevant = mf[mf["miRNA"].isin(exploratory_mirnas)][
            ["miRNA", "family_key"]
        ].drop_duplicates()

        temp = ts.copy()
        temp["family_key"] = temp[mirfam_col].astype(str).str.strip()

        ts_pairs = temp.merge(
            relevant,
            on="family_key",
            how="inner"
        )

        ts_pairs = ts_pairs[
            ts_pairs["Gene"].isin(exploratory_genes)
        ].copy()

ts_pairs["TargetScan"] = True

print("TargetScan candidate pairs:", len(ts_pairs))
display(ts_pairs[["miRNA", "Gene"]].drop_duplicates().head())


Summary Counts: /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets/TargetScan_v8/Summary_Counts.default_predictions.txt.zip
miR Family    : /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets/TargetScan_v8/miR_Family_Info.txt.zip
TargetScan candidate pairs: 8


,miRNA,Gene
7707,hsa-miR-125a-5p,SBNO1
40699,hsa-miR-125a-5p,GK5
46696,hsa-miR-128-3p,PRKX
139326,hsa-miR-128-3p,EIF5
193199,hsa-miR-128-3p,MOB1B


## 5. miRTarBase v9.0

In [14]:
def parse_mirtarbase(path):
    if not path.exists():
        print("miRTarBase file not found; returning empty table.")
        return pd.DataFrame(
            columns=["miRNA", "Gene", "miRTarBase", "Experiments", "Support Type", "PMID"]
        )

    df = pd.read_excel(path)

    def find_any(candidates):
        for cand in candidates:
            for col in df.columns:
                if cand.lower() == str(col).strip().lower():
                    return col
        for cand in candidates:
            for col in df.columns:
                if cand.lower() in str(col).lower():
                    return col
        return None

    mir_col = find_any(["miRNA", "miRNA name"])
    gene_col = find_any(["Target Gene", "Target Gene Symbol", "Gene"])
    exp_col = find_any(["Experiments", "Experiment"])
    sup_col = find_any(["Support Type", "Support"])
    pmid_col = find_any(["References (PMID)", "PMID"])

    if mir_col is None or gene_col is None:
        raise RuntimeError("Could not identify miRTarBase miRNA/target-gene columns.")

    out = pd.DataFrame({
        "miRNA": df[mir_col].astype(str).str.strip(),
        "Gene": df[gene_col].astype(str).str.strip(),
        "Experiments": df[exp_col] if exp_col else np.nan,
        "Support Type": df[sup_col] if sup_col else np.nan,
        "PMID": df[pmid_col] if pmid_col else np.nan,
    })

    out = out[
        out["miRNA"].isin(exploratory_mirnas) &
        out["Gene"].isin(exploratory_genes)
    ].copy()

    out["miRTarBase"] = True
    return out

mirtar_pairs = parse_mirtarbase(MIRTARBASE_FILE)

print("miRTarBase candidate pairs:", len(mirtar_pairs))
display(mirtar_pairs.head())


miRTarBase candidate pairs: 42


,miRNA,Gene,Experiments,Support Type,PMID,miRTarBase
15570,hsa-miR-148b-3p,TEX13A,Microarray,Functional MTI (Weak),17612493,True
18692,hsa-miR-128-3p,BLOC1S2,Sequencing,Functional MTI (Weak),20371350,True
18779,hsa-miR-128-3p,MOB1B,Microarray,Functional MTI (Weak),17612493,True
39343,hsa-miR-328-3p,HIST1H4D,CLASH,Functional MTI (Weak),23622248,True
41315,hsa-miR-125a-5p,PRC1,CLASH,Functional MTI (Weak),23622248,True


## 6. Integrate database support

In [15]:
def standardize_pairs(df, db):
    if df.empty:
        return pd.DataFrame(columns=["miRNA", "Gene", db])

    z = df[["miRNA", "Gene"]].drop_duplicates().copy()
    z[db] = True
    return z

a = standardize_pairs(mirdb_pairs, "miRDB")
b = standardize_pairs(ts_pairs, "TargetScan")
c = standardize_pairs(mirtar_pairs, "miRTarBase")

pairs = a.merge(b, on=["miRNA", "Gene"], how="outer")
pairs = pairs.merge(c, on=["miRNA", "Gene"], how="outer")

for col in ["miRDB", "TargetScan", "miRTarBase"]:
    if col not in pairs.columns:
        pairs[col] = False
    pairs[col] = pairs[col].fillna(False).astype(bool)

pairs["n_databases"] = pairs[
    ["miRDB", "TargetScan", "miRTarBase"]
].sum(axis=1)

pairs["FDR_significant_miRNA"] = pairs["miRNA"].isin(confirmatory_mirnas)

print("Unique supported pairs:", len(pairs))
display(
    pairs["n_databases"]
    .value_counts()
    .sort_index()
    .rename_axis("n_databases")
    .reset_index(name="n_pairs")
)


Unique supported pairs: 26


,n_databases,n_pairs
0,1,23
1,2,3


## 7. Add expression direction

In [16]:
mi_stats = pd.read_csv(
    RESULTS_DIR / "miRNA_limma_full_results.csv"
)[["miRNA", "logFC", "adj.P.Val"]].rename(
    columns={
        "logFC": "miRNA_logFC",
        "adj.P.Val": "miRNA_FDR"
    }
)

mr_stats = pd.read_csv(
    RESULTS_DIR / "mRNA_limma_full_results.csv"
)[["Gene", "logFC", "adj.P.Val"]].rename(
    columns={
        "logFC": "mRNA_logFC",
        "adj.P.Val": "mRNA_FDR"
    }
)

pairs = pairs.merge(mi_stats, on="miRNA", how="left")
pairs = pairs.merge(mr_stats, on="Gene", how="left")

pairs["expression_relationship"] = np.where(
    np.sign(pairs["miRNA_logFC"]) != np.sign(pairs["mRNA_logFC"]),
    "anti-correlated",
    "same-direction"
)

pairs = pairs.sort_values(
    ["FDR_significant_miRNA", "n_databases"],
    ascending=[False, False]
).reset_index(drop=True)

display(pairs.head(30))


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,FDR_significant_miRNA,miRNA_logFC,miRNA_FDR,mRNA_logFC,mRNA_FDR,expression_relationship
0,hsa-miR-128-3p,EIF5,False,True,True,2,False,-1.174363,0.069791,-0.608970,0.965865,same-direction
1,hsa-miR-128-3p,MOB1B,False,True,True,2,False,-1.174363,0.069791,-0.818650,0.965865,same-direction
2,hsa-miR-328-3p,HIST1H4D,False,True,True,2,False,-1.094243,0.129738,-1.003720,0.965865,same-direction
3,hsa-miR-125a-5p,GK5,False,True,False,1,False,-0.729913,0.510138,-0.602703,0.965865,same-direction
4,hsa-miR-125a-5p,PRC1,False,False,True,1,False,-0.729913,0.510138,-0.587820,0.965865,same-direction
5,hsa-miR-125a-5p,SBNO1,False,True,False,1,False,-0.729913,0.510138,-0.588660,0.965865,same-direction
6,hsa-miR-128-3p,BLOC1S2,False,False,True,1,False,-1.174363,0.069791,-0.603343,0.965865,same-direction
7,hsa-miR-128-3p,CISD1,False,False,True,1,False,-1.174363,0.069791,0.616583,0.965865,anti-correlated
8,hsa-miR-128-3p,PRKX,False,True,False,1,False,-1.174363,0.069791,-0.587287,0.965865,same-direction
9,hsa-miR-128-3p,SBNO1,False,True,False,1,False,-1.174363,0.069791,-0.588660,0.965865,same-direction


## 8. Main integration results

In [17]:
fdr_mirna_pairs = pairs[
    pairs["FDR_significant_miRNA"]
].copy()

multi_db_pairs = pairs[
    pairs["n_databases"] >= 2
].copy()

anti_pairs = pairs[
    pairs["expression_relationship"] == "anti-correlated"
].copy()

multi_db_anti = pairs[
    (pairs["n_databases"] >= 2) &
    (pairs["expression_relationship"] == "anti-correlated")
].copy()

summary = pd.DataFrame({
    "Metric": [
        "All database-supported pairs",
        "Pairs involving FDR-significant miRNA",
        "Pairs supported by >=2 databases",
        "Anti-correlated pairs",
        "Pairs with >=2 DB support AND anti-correlation",
    ],
    "Value": [
        len(pairs),
        len(fdr_mirna_pairs),
        len(multi_db_pairs),
        len(anti_pairs),
        len(multi_db_anti),
    ]
})

display(summary)

print("\nMulti-database-supported pairs")
display(multi_db_pairs)

print("\nAnti-correlated pairs")
display(anti_pairs)

print("\nFDR-significant miRNA pairs")
display(fdr_mirna_pairs)


,Metric,Value
0,All database-supported pairs,26
1,Pairs involving FDR-significant miRNA,0
2,Pairs supported by >=2 databases,3
3,Anti-correlated pairs,3
4,Pairs with >=2 DB support AND anti-correlation,0



Multi-database-supported pairs


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,FDR_significant_miRNA,miRNA_logFC,miRNA_FDR,mRNA_logFC,mRNA_FDR,expression_relationship
0,hsa-miR-128-3p,EIF5,False,True,True,2,False,-1.174363,0.069791,-0.60897,0.965865,same-direction
1,hsa-miR-128-3p,MOB1B,False,True,True,2,False,-1.174363,0.069791,-0.81865,0.965865,same-direction
2,hsa-miR-328-3p,HIST1H4D,False,True,True,2,False,-1.094243,0.129738,-1.00372,0.965865,same-direction



Anti-correlated pairs


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,FDR_significant_miRNA,miRNA_logFC,miRNA_FDR,mRNA_logFC,mRNA_FDR,expression_relationship
7,hsa-miR-128-3p,CISD1,False,False,True,1,False,-1.174363,0.069791,0.616583,0.965865,anti-correlated
11,hsa-miR-148b-3p,TEX13A,False,False,True,1,False,-0.908403,0.109901,0.656873,0.965865,anti-correlated
17,hsa-miR-6779-5p,CBY3,False,False,True,1,False,-0.681383,0.178395,0.601470,0.965865,anti-correlated



FDR-significant miRNA pairs


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,FDR_significant_miRNA,miRNA_logFC,miRNA_FDR,mRNA_logFC,mRNA_FDR,expression_relationship


## 9. Export main-analysis results

In [18]:
all_out = RESULTS_DIR / "miRNA_mRNA_all_supported_pairs.csv"
multi_out = RESULTS_DIR / "miRNA_mRNA_multi_database_pairs.csv"
anti_out = RESULTS_DIR / "miRNA_mRNA_anti_correlated_pairs.csv"
fdr_out = RESULTS_DIR / "miRNA_mRNA_FDR_miRNA_pairs.csv"
summary_out = RESULTS_DIR / "miRNA_mRNA_integration_summary.csv"
xlsx_out = RESULTS_DIR / "miRNA_mRNA_integration.xlsx"

pairs.to_csv(all_out, index=False)
multi_db_pairs.to_csv(multi_out, index=False)
anti_pairs.to_csv(anti_out, index=False)
fdr_mirna_pairs.to_csv(fdr_out, index=False)
summary.to_csv(summary_out, index=False)

with pd.ExcelWriter(xlsx_out) as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    pairs.to_excel(writer, sheet_name="All pairs", index=False)
    multi_db_pairs.to_excel(writer, sheet_name="Multi-database", index=False)
    anti_pairs.to_excel(writer, sheet_name="Anti-correlated", index=False)
    fdr_mirna_pairs.to_excel(writer, sheet_name="FDR miRNA pairs", index=False)

print("Saved main integration outputs to:")
print(RESULTS_DIR)


Saved main integration outputs to:
/Users/jihopark/Desktop/MCDA_revision_final/results


## Interpretation

The miRNA–mRNA integration is treated as a downstream exploratory analysis.

- **Multi-database support** strengthens annotation-level evidence but does not prove regulation in this study.
- **Anti-correlated expression** is directionally compatible with canonical miRNA-mediated repression but is not causal evidence.
- The BH-FDR-significant miRNA is evaluated separately.
- If no pair satisfies both multi-database support and anti-correlated expression, this absence should be reported transparently.
